# Simple MS Simulator — Phase 1, step 5: ICEBERG throughput on Kaggle T4

**Generated by `build/mk_simple_sim_grid.py` — do not hand-edit.**

Settings: **GPU T4 ×2**, **Internet OFF**. Inputs: the competition data, **casmi26-iceberg-offline**
(wheelhouse + checkpoints from Phase 0) and the COCONUT dataset phase1 uses (`coco_meta.pkl`,
`coco_mass.npy`). ~20 min.

The request is built here, from the same inputs the Colab notebook uses: 2,000 candidates drawn from
the pool ranges phase1 would score for the `enveda-np-examples` molecules (so the mass mix matches the
real on-the-fly load), cycling `[M+H]+`/`[M-H]-` × 20/40/60 eV. The worker is the current one
(precursor + 2 Da cut), embedded from `build/iceberg_worker.py`.

Runs: 1 T4 at batch 64 and 128, then 2 T4s at the better batch size (one worker per GPU, half the
rows each, concurrently). Output `/kaggle/working/t4.json` → copy it to Drive `_simlib/grid/t4.json`.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

def find_one(pattern, is_dir=False):
    hits = [h for h in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
            if (os.path.isdir(h) if is_dir else os.path.isfile(h))]
    assert hits, f"{pattern} not found under /kaggle/input -- is the dataset attached?"
    return sorted(hits, key=len)[0]

UP = os.path.dirname(find_one("wheelhouse", is_dir=True))
WH = os.path.join(UP, "wheelhouse")
MAN = json.load(open(os.path.join(UP, "manifest.json")))
TRAIN = find_one("train.parquet")
COCO = os.path.dirname(find_one("coco_meta.pkl"))
WORK = Path("/kaggle/working/t4")
WORK.mkdir(parents=True, exist_ok=True)
N_REQ = 2000
SEED = 0
GPUS = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                      capture_output=True, text=True).stdout.strip().splitlines()
print(f"dataset {UP}\ntrain {TRAIN}\ncoconut {COCO}\nGPUs {GPUS}; vCPU {os.cpu_count()}")
RESULT = {"gpus": GPUS, "vcpu": os.cpu_count(), "manifest": MAN, "n_requests": N_REQ}

## Shared helpers (embedded from `build/simsim_common.py`)

In [ ]:
"""Shared helpers for the specs_simple_ms_simulator Phase-1 notebooks.

Inlined verbatim into the generated notebooks (at the SIMSIM_COMMON placeholder) by build/mk_simple_sim_grid.py,
and imported directly by build/test_simple_sim_grid.py. Pure numpy / pandas / pyarrow: no RDKit, no
torch, no ms-pred. `prep` / `esim` call phase1's numba kernels (`_clean`, `entropy_sim`), which the
notebooks embed in an earlier cell; the tests inject them.

Lifted from build/simple_sim_gate.py (sh, sha1, formula_mass, the POOL build, to_model_ce, pb_sf, the
worker call) and from phase1 (ADDUCTS physics, the stage-A/B window logic). The gate notebook itself is
left untouched.
"""
import hashlib
import json
import os
import pickle
import re
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

TEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]

# phase1 CFG values (spectrum cleaning and the candidate window) -- unchanged
FLOOR, TOPK, POWER, ENTW, TOL = 0.002, 64, 1.0, True, 0.01
PPM_WIN, PPM_CENTER, PPM_FALLBACK = 8.5, 1.45, 30.0
PPM_EXPAND = (8.5, 20.0, 45.0, 80.0)
TOPN = 25

# --- adduct physics: phase1 cell 3, restricted to the ten test adducts (all n = 1, z = 1) -----------
_M = dict(H=1.00782503207, N=14.0030740048, O=15.9949146196, C=12.0, Na=22.9897692809,
          K=38.96370668, Cl=34.96885268)
_E = 0.00054857990
_PROTON = _M["H"] - _E
_H2O = 2 * _M["H"] + _M["O"]
_NH4 = _M["N"] + 4 * _M["H"]
_FORMATE = _M["C"] + 2 * _M["H"] + 2 * _M["O"]
ADDUCT_DELTA = {
    "[M+H]+": _PROTON, "[M+NH4]+": _NH4 - _E, "[M+Na]+": _M["Na"] - _E, "[M+K]+": _M["K"] - _E,
    "[M-H2O+H]+": _PROTON - _H2O, "[M-2H2O+H]+": _PROTON - 2 * _H2O,
    "[M-H]-": -_PROTON, "[M-H2O-H]-": -_PROTON - _H2O, "[M+CH2O2-H]-": _FORMATE - _PROTON,
    "[M+Cl]-": _M["Cl"] + _E,
}


def neutral_mass1(mz, adduct):
    """ONE precursor m/z -> neutral mass (phase1 neutral_mass, test adducts only; NaN elsewhere).

    Not called `neutral_mass`: the submission notebook also embeds phase1's vectorised neutral_mass.
    """
    d = ADDUCT_DELTA.get(adduct)
    return float(mz) - d if d is not None else np.nan


def precursor_of(mass, adduct):
    """neutral exact mass -> theoretical precursor m/z (the inverse of neutral_mass1)."""
    return np.asarray(mass, dtype=np.float64) + ADDUCT_DELTA[adduct]


# --- label formula -> exact neutral mass (as phase1 / the gate; no RDKit) ---------------------------
_MONO = {"C": 12.0, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221, "S": 31.97207069,
         "P": 30.97376151, "F": 18.99840320, "Cl": 34.96885271, "Br": 78.9183376, "I": 126.904468,
         "Si": 27.9769265327, "Se": 79.9165196, "B": 11.0093055, "Na": 22.98976928, "K": 38.96370649,
         "Fe": 55.9349375, "D": 2.0141017778, "As": 74.9215965, "Li": 7.0160034, "Sn": 119.9021947}
_TOK = re.compile(r"([A-Z][a-z]?)(\d*)")


def formula_mass(f):
    if not isinstance(f, str):
        return np.nan
    s, pos = 0.0, 0
    for m in _TOK.finditer(f):
        if m.start() != pos:
            return np.nan
        pos = m.end()
        if m.group(1) not in _MONO:
            return np.nan
        s += _MONO[m.group(1)] * (int(m.group(2)) if m.group(2) else 1)
    return s if pos == len(f) else np.nan


# --- shell / files ------------------------------------------------------------------------------------
def sh(cmd, timeout=None, env=None):
    print("$", cmd if len(cmd) < 300 else cmd[:300] + " ...", flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout, env=env)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout


def sha1(path, chunk=1 << 20):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# --- ICEBERG worker call (the worker runs in its own venv; files only) -------------------------------
def to_model_ce(ce_ev, prec, mode):
    return ce_ev if mode == "ev" else ce_ev * 500.0 / prec     # ms-pred's own ev_to_nce


def write_request(req, path):
    req[["key", "smiles", "adduct", "precursor_mz", "ce", "instrument"]].to_csv(path, sep="\t", index=False)


def worker_cmd(py, worker, gen, inten, inp, out, device="cuda", batch_size=128, sparse_k=100, prec_margin=2.0):
    return [str(py), str(worker), "--gen", str(gen), "--inten", str(inten), "--inp", str(inp), "--out", str(out),
            "--device", device, "--batch-size", str(batch_size), "--sparse-k", str(sparse_k),
            "--prec-margin", str(prec_margin)]


def read_response(out, n):
    """-> spectra [(mz, it)], ok, err, timing"""
    z = np.load(out)
    # read each array ONCE: NpzFile re-reads (and re-allocates) an array on every z[key] access, and a
    # slice keeps its whole copy alive -- per-row access held n full copies (the Phase-1 Colab OOM)
    off, mz, it = z["off"], z["mz"], z["it"]
    assert len(off) == n + 1, f"{out}: {len(off) - 1} rows, expected {n}"
    spectra = [(mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]) for i in range(n)]
    return spectra, z["ok"], z["err"], json.loads(str(z["timing"]))


def run_worker(req, work, tag, py, worker, gen, inten, env, **kw):
    """-> spectra, ok, err, timing, wall_s"""
    inp, out = Path(work) / f"req_{tag}.tsv", Path(work) / f"resp_{tag}.npz"
    write_request(req, inp)
    t = time.time()
    r = subprocess.run(worker_cmd(py, worker, gen, inten, inp, out, **kw), text=True, capture_output=True,
                       env=env, timeout=12 * 3600)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-4000:])
        raise RuntimeError(f"worker failed on {tag}")
    spectra, ok, err, timing = read_response(out, len(req))
    return spectra, ok, err, timing, time.time() - t


def err_counts(err):
    return pd.Series([str(e).split(":")[0] for e in err if e]).value_counts().to_dict()


# --- spectra -----------------------------------------------------------------------------------------
def prep(mz, it, prec):
    """phase1 cleaning after the precursor + 2 Da cut. Needs the phase1 kernels in globals."""
    mz = np.asarray(mz, np.float32)
    it = np.asarray(it, np.float32)
    keep = mz <= prec + 2.0
    return _clean(mz[keep], it[keep], FLOOR, TOPK, POWER, ENTW)  # noqa: F821 (phase1 kernel cell)


def esim(a, b):
    if len(a[0]) == 0 or len(b[0]) == 0:
        return 0.0
    return float(entropy_sim(a[0], a[1], b[0], b[1], TOL))  # noqa: F821 (phase1 kernel cell)


def sum_merge(specs, decimals=4):
    """Merge per-CE predicted spectra the way a merged acquisition sums them.

    Each input is (mz, it) with base peak 1. Peaks with the same m/z (rounded to `decimals`; the same
    fragment formula gives the same m/z at every CE) are summed, then the result is renormalised to
    base 1 and sorted by m/z. Empty inputs are ignored.
    """
    parts = [(np.asarray(m, np.float64), np.asarray(i, np.float64)) for m, i in specs if len(m)]
    if not parts:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mz = np.concatenate([m for m, _ in parts])
    it = np.concatenate([i / i.max() for _, i in parts])
    key, inv = np.unique(np.round(mz, decimals), return_inverse=True)
    s = np.bincount(inv, weights=it)
    m = np.bincount(inv, weights=mz) / np.bincount(inv)
    o = np.argsort(m)
    return m[o].astype(np.float32), (s[o] / s.max()).astype(np.float32)


def rank_of_truth(sims):
    """sims[0] = truth, sims[1:] = decoys. Ties count half; a tied truth is not top-1."""
    sims = np.asarray(sims, dtype=np.float64)
    t, dec = sims[0], sims[1:]
    gt, eq = int((dec > t).sum()), int((dec == t).sum())
    return 1 + gt + eq / 2, (gt == 0 and eq == 0)


def pb_sf(ps, k):
    """P(X >= k), X = sum of independent Bernoulli(p_i) -- exact Poisson-binomial by DP."""
    d = np.zeros(len(ps) + 1)
    d[0] = 1.0
    for p in ps:
        d[1:] = d[1:] * (1 - p) + d[:-1] * p
        d[0] *= (1 - p)
    return float(d[k:].sum())


# --- collision-energy grid ----------------------------------------------------------------------------
def ce_tuple(v):
    """collision_energy_ev cell (list / array / None) -> tuple of floats rounded to 0.1 eV."""
    if v is None:
        return ()
    try:
        return tuple(round(float(x), 1) for x in v)
    except TypeError:
        return ()


def snap_ce(ce, grid, tol):
    """Nearest grid CE within tol (eV), else None. Ties go to the lower CE."""
    if not grid or not np.isfinite(ce):
        return None
    g = min(sorted(grid), key=lambda x: abs(x - ce))
    return float(g) if abs(g - ce) <= tol + 1e-9 else None


def spectrum_conditions(adduct, ces, grid, tol, covered):
    """One spectrum -> set of (adduct, ce, on_grid) conditions that need a simulated spectrum.

    Uncovered adducts need none (MetFrag-lite). A merged multi-CE spectrum needs each constituent CE,
    which is why merges add no grid conditions. CE <= 0 constituents (a few libraries list 0) are skipped.
    """
    if adduct not in covered:
        return set()
    out = set()
    for ce in ces:
        if not np.isfinite(ce) or ce <= 0:
            continue
        g = snap_ce(ce, grid, tol)
        out.add((adduct, g if g is not None else round(float(ce), 1), g is not None))
    return out


def choose_tolerance(median_by_delta, min_sim=0.90):
    """Largest delta (eV) whose median sim(pred@c, pred@c+delta) is >= min_sim; 0.0 if none."""
    ok = [d for d, m in median_by_delta.items() if np.isfinite(m) and m >= min_sim]
    return float(max(ok)) if ok else 0.0


def base_ce_set(ce_tuples, min_cover=0.95):
    """Distinct single CEs covering >= min_cover of single-CE spectra, plus every merged constituent."""
    singles = pd.Series([t[0] for t in ce_tuples if len(t) == 1 and t[0] > 0])
    keep = set()
    if len(singles):
        vc = singles.value_counts()
        cum = vc.cumsum() / vc.sum()
        keep = set(vc.index[: int((cum < min_cover).sum()) + 1])
    for t in ce_tuples:
        if len(t) > 1:
            keep |= {c for c in t if c > 0}
    return sorted(float(c) for c in keep)


# --- candidate pool and windows ------------------------------------------------------------------------
def build_pool(coco_dir, train_path):
    """COCONUT + train structures, deduplicated on ik14, sorted by exact neutral mass (as the gate).

    Columns: k14, smiles, mass, has_real (the structure has >= 1 training spectrum).
    """
    coco_dir = Path(coco_dir)
    cm = pickle.load(open(coco_dir / "coco_meta.pkl", "rb"))
    cmass = np.load(coco_dir / "coco_mass.npy")
    tr = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles", "molecular_formula"]).to_pandas()
    tr = tr.dropna(subset=["inchikey14", "normalized_smiles"]).drop_duplicates("inchikey14")
    fm = {f: formula_mass(f) for f in set(tr.molecular_formula.astype(object).tolist())}
    tr_mass = np.array([fm.get(f, np.nan) for f in tr.molecular_formula.astype(object)], dtype=np.float64)
    pool = pd.DataFrame({
        "k14": np.concatenate([[str(k)[:14] for k in cm["keys"]], tr.inchikey14.astype(str).to_numpy()]),
        "smiles": np.concatenate([np.asarray(cm["smiles"], dtype=object),
                                  tr.normalized_smiles.astype(object).to_numpy()]),
        "mass": np.concatenate([cmass.astype(np.float64), tr_mass]),
    })
    n_raw = len(pool)
    pool = (pool[np.isfinite(pool.mass) & pool.smiles.notna()].drop_duplicates("k14")
            .sort_values("mass", kind="stable").reset_index(drop=True))
    pool["has_real"] = pool.k14.isin(set(tr.inchikey14.astype(str)))
    print(f"pool {n_raw:,} rows -> {len(pool):,} unique ik14 with a finite mass "
          f"(COCONUT {len(cmass):,}, train {len(tr):,})")
    return pool


def window(pm, target, ppm):
    """(lo, hi) index range of the mass-sorted pool within +-ppm of target (phase1 CandidatePool.window)."""
    lo = int(np.searchsorted(pm, target * (1 - ppm / 1e6), "left"))
    hi = int(np.searchsorted(pm, target * (1 + ppm / 1e6), "right"))
    return lo, hi


def scored_range(pm, target, win=PPM_WIN, fallback=PPM_FALLBACK, expand=PPM_EXPAND, topn=TOPN):
    """Pool range whose candidates phase1's stages A + B actually score (stage C only mass-sorts).

    Stage A: the +-win window, or +-fallback if that is empty. Stage B: while fewer than topn candidates
    (pool keys are unique), every wider tier's new candidates are scored. The windows are nested, so the
    scored set is one contiguous range.
    """
    lo, hi = window(pm, target, win)
    if hi == lo:
        lo, hi = window(pm, target, fallback)
    if hi - lo < topn:
        for ppm in expand[1:]:
            a, b = window(pm, target, ppm)
            lo, hi = min(lo, a), max(hi, b)
            if hi - lo >= topn:
                break
    return lo, hi


def query_target(precs, adducts):
    """Molecule -> centred neutral target mass (phase1: median over spectra, divided by 1 + PPM_CENTER)."""
    nms = np.array([neutral_mass1(p, a) for p, a in zip(precs, adducts)], dtype=np.float64)
    nms = nms[np.isfinite(nms)]
    return float(np.median(nms)) / (1.0 + PPM_CENTER / 1e6) if len(nms) else np.nan


# --- projection and decision ---------------------------------------------------------------------------
def on_the_fly(conds, precomputed_adducts):
    """Conditions left for Kaggle: off-grid ones, and on-grid ones whose adduct was not precomputed."""
    return [c for c in conds if not (c[2] and c[0] in precomputed_adducts)]


def bootstrap_total(per_mol, n_mol=400, n_boot=2000, seed=0):
    """Total over n_mol molecules resampled from per_mol -> dict(mean, p50, p95)."""
    x = np.asarray(per_mol, dtype=np.float64)
    if len(x) == 0:
        return dict(mean=0.0, p50=0.0, p95=0.0)
    rng = np.random.default_rng(seed)
    tot = x[rng.integers(0, len(x), size=(n_boot, n_mol))].sum(1)
    return dict(mean=float(x.mean() * n_mol), p50=float(np.percentile(tot, 50)), p95=float(np.percentile(tot, 95)))


def kaggle_hours(load, rate, n_mol, t_base_s, fixed_s, simlib_gb, load_s_per_gb, worker_calls, startup_s):
    """Projected Kaggle wall time (h): fixed setup + per-molecule base cost + simlib load + on-the-fly sims."""
    sim_s = (load / rate if load > 0 else 0.0) + (worker_calls * startup_s if load > 0 else 0.0)
    return (fixed_s + n_mol * t_base_s + simlib_gb * load_s_per_gb + sim_s) / 3600.0


def decide(options, budget_h):
    """Cheapest option (colab_h, then name) whose kaggle_h <= budget_h; None if none fits (-> STOP)."""
    fit = [o for o in options if o["kaggle_h"] <= budget_h]
    return min(fit, key=lambda o: (o["colab_h"], o["name"])) if fit else None


# --- submission (Phase 4, option A: every covered condition simulated on the fly) ---------------------
MERGE_RULES = ("max", "sum", "mid")


def spectrum_ces(ces, rule, default):
    """One query spectrum's CE list -> (CEs to simulate, defaulted).

    CE <= 0 constituents are dropped (a few lists start at 0). `max` / `sum` simulate every constituent;
    `mid` simulates the mean of the list once. No usable CE -> [default], defaulted = True.
    """
    assert rule in MERGE_RULES, rule
    v = [round(float(c), 1) for c in ces if np.isfinite(c) and c > 0]
    if not v:
        return [float(default)], True
    if rule == "mid":
        return [round(float(np.mean(v)), 1)], False
    return sorted(set(v)), False


def sim_priority(lv, fr, ppm):
    """Per-candidate simulation priority (0 = first): lv desc, fr desc, |ppm| asc -- the watchdog's cap order."""
    o = np.lexsort((np.asarray(ppm, np.float64), -np.asarray(fr, np.float64), -np.asarray(lv, np.float64)))
    r = np.empty(len(o), np.int64)
    r[o] = np.arange(len(o))
    return r


REQ_COLS = ["key", "smiles", "adduct", "precursor_mz", "ce_ev", "prio", "ce", "instrument"]


def build_requests(parts, instrument="QTOF", ce_mode="ev"):
    """Molecule parts -> one worker request per distinct (key, adduct, ce_ev), sorted by priority.

    Each part is a dict with arrays key / smiles / mass / prio (one entry per candidate to simulate) and
    conds, a list of (adduct, ce_ev). The precursor is the candidate's theoretical m/z (exact mass +
    adduct delta, as the G2b gate), so a request depends on the candidate alone and is shared by every
    spectrum and molecule that needs it; a shared request keeps its best (lowest) priority.
    """
    frames = []
    for p in parts:
        if not len(p["key"]) or not p["conds"]:
            continue
        mass = np.asarray(p["mass"], np.float64)
        for adduct, ce in p["conds"]:
            frames.append(pd.DataFrame({"key": np.asarray(p["key"], dtype=object),
                                        "smiles": np.asarray(p["smiles"], dtype=object), "adduct": adduct,
                                        "precursor_mz": precursor_of(mass, adduct), "ce_ev": float(ce),
                                        "prio": np.asarray(p["prio"], np.int64)}))
    if not frames:
        return pd.DataFrame({c: [] for c in REQ_COLS})
    req = (pd.concat(frames, ignore_index=True)
           .sort_values(["prio", "key", "adduct", "ce_ev"], kind="stable")
           .drop_duplicates(["key", "adduct", "ce_ev"], keep="first").reset_index(drop=True))
    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), ce_mode)
    req["instrument"] = instrument
    return req[REQ_COLS]


def rows_that_fit(remaining_s, rate, startup_s):
    """How many worker rows one more call can take before the deadline (0 if none)."""
    return max(0, int((remaining_s - startup_s) * rate))


def spectrum_sv(q, sims, rule):
    """Simulated-library score of ONE prepped query spectrum against one candidate.

    sims: the candidate's raw predictions (mz, it, prec) at each of the spectrum's CEs, None where it was
    not simulated (failed, capped). max / mid: best esim over the CEs; sum: esim against the renormalised
    sum of the available CEs (a merged acquisition). Nothing available -> 0.
    """
    have = [s for s in sims if s is not None and len(s[0])]
    if not have or len(q[0]) == 0:
        return 0.0
    if rule == "sum":
        mz, it = sum_merge([(m, i) for m, i, _ in have])
        return esim(q, prep(mz, it, have[0][2]))
    return max(esim(q, prep(m, i, p)) for m, i, p in have)


def rank_two_tier(lv, sv, fr, ppm, tau):
    """Order (indices) of one candidate set.

    Tier 1 = real-library match lv >= tau, by lv desc (then sv, fr, |ppm|). Tier 2 = the rest, by sv desc,
    fr desc, |ppm| asc. A missing sv / fr (NaN) counts 0; a missing ppm sorts last.
    """
    lv, sv, fr = (np.nan_to_num(np.asarray(x, np.float64), nan=0.0) for x in (lv, sv, fr))
    ppm = np.nan_to_num(np.asarray(ppm, np.float64), nan=np.inf)
    t1, t2 = np.flatnonzero(lv >= tau), np.flatnonzero(lv < tau)
    o1 = t1[np.lexsort((ppm[t1], -fr[t1], -sv[t1], -lv[t1]))]
    o2 = t2[np.lexsort((ppm[t2], -fr[t2], -sv[t2]))]
    return np.concatenate([o1, o2]).astype(np.int64)

## Offline install (as the Phase-0 G5 notebook) + worker + SHA1

In [ ]:
VENV = "/kaggle/working/iceberg_venv"
PY = f"{VENV}/bin/python"
site_pkg = os.path.join(VENV, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
ENV["PYTHONPATH"] = site_pkg
ENV["DGLBACKEND"] = "pytorch"

t0 = time.time()
if not os.path.exists(PY):
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV], check=True)
os.makedirs(site_pkg, exist_ok=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", site_pkg,
                    *sorted(glob.glob(f"{WH}/*.whl"))], capture_output=True, text=True, env=ENV)
assert r.returncode == 0, r.stderr[-3000:]
RESULT["install_s"] = round(time.time() - t0, 1)

WORKER_SRC = '"""ICEBERG forward-simulation worker for specs_simple_ms_simulator.\n\nRuns INSIDE the isolated ms-pred environment (its own torch 2.6 / DGL / rdkit 2025.03), never in\nthe notebook\'s main interpreter -- that keeps RDKit 2026.03.3 (the metric\'s version) untouched.\nThe notebook talks to it through files only:\n\n    request tsv      : key, smiles, adduct (CASMI spelling), precursor_mz, ce, instrument\n                       (TSV, not parquet: the ms-pred environment ships no pyarrow)\n    response npz     : off (n+1,) int64, mz / it float32 flat, ok bool, err str, timing json,\n                       above_n int32 / above_share float32 / above_off float32 (per row, see below)\n\nPeaks are the top `--sparse-k` by intensity, sorted by m/z, base peak = 1.0. m/z values are the\ncharged fragment masses (ICEBERG applies the adduct shift itself).\n\nPeaks above precursor_mz + `--prec-margin` (2 Da) are dropped BEFORE the top-k, the same cleaning the\norganisers applied to the test spectra. They exist because the MassSpecGym inten checkpoints have\n`include_unshifted_mz=True`: every fragment is also emitted without the adduct shift, and for\nwater-loss adducts the unshifted root lands 17-35 Da above the precursor. What was dropped is\nrecorded per row: above_n = peak count, above_share = share of raw intensity, above_off = the\nlargest dropped m/z minus the precursor (NaN when nothing was dropped). A row left with no peaks\ngets err "empty_after_prec_cut".\n\nReused unchanged by Phase 0 (gate), Phase 2 (precompute) and Phase 4 (Kaggle on-the-fly).\n\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --inp req.tsv --out resp.npz\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --hparams-only --out hp.json\n"""\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom rdkit import Chem, RDLogger\n\nimport importlib.abc  # noqa: E402\nimport importlib.machinery  # noqa: E402\nimport os  # noqa: E402\nimport sys  # noqa: E402\nimport types  # noqa: E402\n\nSTUBBED = []\n\n\nclass _Missing:\n    """Stands in for a name from an ms_pred module the package does not ship; fails only if actually used."""\n\n    def __init__(self, name):\n        self._name = name\n\n    def __call__(self, *a, **k):\n        raise RuntimeError(f"{self._name} is not shipped with this ms_pred copy")\n\n    def __getattr__(self, attr):\n        raise RuntimeError(f"{self._name}.{attr} is not shipped with this ms_pred copy")\n\n\nclass _StubMissingMsPred(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    """The casmi26-iceberg package ships a trimmed ms_pred (inference modules only) whose package __init__s still\n    import training-side modules (e.g. nn_utils.tune_utils). Last on sys.meta_path, so it only answers for an\n    ms_pred submodule no real finder can find, with an empty module."""\n\n    def find_spec(self, name, path=None, target=None):\n        return importlib.machinery.ModuleSpec(name, self) if name.startswith("ms_pred.") else None\n\n    def create_module(self, spec):\n        m = types.ModuleType(spec.name)\n        m.__all__ = []\n\n        def __getattr__(attr, _n=spec.name):\n            if attr.startswith("__"):\n                raise AttributeError(attr)\n            return _Missing(f"{_n}.{attr}")\n        m.__getattr__ = __getattr__\n        return m\n\n    def exec_module(self, module):\n        STUBBED.append(module.__name__)\n\n\nclass _Anything:\n    """Permissive placeholder served by an absent third-party module (import-time use only; ice_runner.py\'s)."""\n\n    def __init__(self, *a, **k):\n        pass\n\n    def __call__(self, *a, **k):\n        return _Anything()\n\n    def __getattr__(self, k):\n        if k.startswith("__"):\n            raise AttributeError(k)\n        return _Anything()\n\n    def __getitem__(self, k):\n        return _Anything()\n\n    def __setitem__(self, k, v):\n        pass\n\n    def __iter__(self):\n        return iter(())\n\n    def __bool__(self):\n        return False\n\n\nclass _StubModule(types.ModuleType):\n    def __getattr__(self, k):\n        if k.startswith("__"):\n            raise AttributeError(k)\n        v = type(k, (_Anything,), {}) if k[:1].isupper() else _Anything()\n        setattr(self, k, v)\n        return v\n\n\nclass _StubThirdParty(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    """Serves stub modules (and any submodule) for third-party roots that ms_pred imports at module level but never\n    uses on the predict_mol path, when the image lacks them (multiprocess, h5py, pygmtools on the cp313 image)."""\n\n    def __init__(self, roots):\n        self.roots = set(roots)\n\n    def find_spec(self, name, path=None, target=None):\n        if name.split(".")[0] not in self.roots:\n            return None\n        return importlib.machinery.ModuleSpec(name, self, is_package=True)\n\n    def create_module(self, spec):\n        m = _StubModule(spec.name)\n        m.__path__ = []\n        return m\n\n    def exec_module(self, module):\n        pass\n\n\n# the casmi26-iceberg ice_runner.py list: imported at ms_pred module level, unused by ICEBERG inference\nOPTIONAL = ("pygmtools", "h5py", "multiprocess", "platformdirs")\n\nif os.environ.get("CASMI_ICE_PKG") == "1":\n    sys.meta_path.append(_StubMissingMsPred())\n    _absent = []\n    for _m in OPTIONAL:\n        try:\n            importlib.import_module(_m)\n        except BaseException:\n            for _k in [k for k in sys.modules if k.split(".")[0] == _m]:\n                del sys.modules[_k]\n            _absent.append(_m)\n    if _absent:\n        sys.meta_path.insert(0, _StubThirdParty(_absent))\n        print(f"[worker] absent third-party modules, stubbed: {_absent}", file=sys.stderr, flush=True)\n\nimport ms_pred.common as common  # noqa: E402\nimport ms_pred.common.chem_utils as _chem_utils  # noqa: E402\nfrom ms_pred.iceberg import gen_model, inten_model, joint_model  # noqa: E402\n\nif STUBBED:\n    print(f"[worker] ms_pred modules not shipped, stubbed: {STUBBED}", file=sys.stderr, flush=True)\n\n# predict_mol\'s InChI round trip writes a constant tautomer-transform file under platformdirs\' user_cache_dir:\n# point it at a writable temp dir (also needed when platformdirs itself is stubbed)\nimport tempfile  # noqa: E402\n_CACHE = os.path.join(tempfile.gettempdir(), "ice_ms_pred_cache")\n_chem_utils.user_cache_dir = lambda name, *a, **k: os.path.join(_CACHE, name)\n\nimport rdkit  # noqa: E402\nprint(f"[worker] rdkit {rdkit.__version__} at {os.path.dirname(rdkit.__file__)} | torch {torch.__version__}",\n      file=sys.stderr, flush=True)\nif os.environ.get("CASMI_ICE_PKG") == "1" and not rdkit.__version__.startswith("2025.03"):\n    raise RuntimeError(f"rdkit {rdkit.__version__}: the casmi26-iceberg checkpoints need 2025.03 "\n                       "(2026.03 kekulizes differently) -- the ice site is not first on PYTHONPATH")\n\nRDLogger.DisableLog("rdApp.*")\n\n# CASMI adduct spelling -> ms-pred spelling (only where they differ)\nADDUCT_MAP = {"[M-2H2O+H]+": "[M+H-2H2O]+", "[M+CH2O2-H]-": "[M+CHO2]-"}\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--gen", required=True)\n    p.add_argument("--inten", required=True)\n    p.add_argument("--inp")\n    p.add_argument("--out", required=True)\n    p.add_argument("--device", default="cuda")\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--max-nodes", type=int, default=100)\n    p.add_argument("--sparse-k", type=int, default=100)\n    p.add_argument("--threshold", type=float, default=0.0)\n    p.add_argument("--prec-margin", type=float, default=2.0)\n    p.add_argument("--hparams-only", action="store_true")\n    return p.parse_args()\n\n\ndef _jsonable(d):\n    out = {}\n    for k, v in dict(d).items():\n        try:\n            json.dumps(v)\n            out[k] = v\n        except TypeError:\n            out[k] = str(v)\n    return out\n\n\n# PyTorch 2.6 defaults weights_only=True, which breaks legacy Lightning checkpoints containing PosixPath/hparams.\ntry:\n    from pathlib import PosixPath, WindowsPath\n    torch.serialization.add_safe_globals([PosixPath, WindowsPath])\nexcept Exception:\n    pass\n\n_orig_torch_load = torch.load\ndef _safe_torch_load(*load_args, **load_kwargs):\n    if "weights_only" not in load_kwargs:\n        load_kwargs["weights_only"] = False\n    return _orig_torch_load(*load_args, **load_kwargs)\ntorch.load = _safe_torch_load\n\n\nHP_KEYS = ("hyper_parameters", "hparams", "hp", "config", "kwargs", "args")\nSD_KEYS = ("state_dict", "model_state_dict", "model", "weights")\n\n\ndef load_model(cls, path):\n    """A Lightning checkpoint (.ckpt), or a plain torch file (.pt, e.g. the casmi26-iceberg package): a pickled\n    module, or a dict holding hyper-parameters + state_dict."""\n    if str(path).endswith(".pt"):   # the casmi26-iceberg layout: {hyper_parameters, state_dict}, as ice_runner.py loads it\n        obj = torch.load(path, map_location="cpu")\n        if isinstance(obj, dict) and "hyper_parameters" in obj and "state_dict" in obj:\n            m = cls(**obj["hyper_parameters"])\n            m.load_state_dict(obj["state_dict"], strict=True)\n            return m\n    try:\n        try:\n            return cls.load_from_checkpoint(path, map_location="cpu", weights_only=False)\n        except TypeError:\n            return cls.load_from_checkpoint(path, map_location="cpu")\n    except Exception as e:\n        lightning_err = e\n    obj = torch.load(path, map_location="cpu")\n    if isinstance(obj, torch.nn.Module):\n        return obj\n    if isinstance(obj, dict):\n        hp = next((obj[k] for k in HP_KEYS if k in obj), None)\n        sd = next((obj[k] for k in SD_KEYS if k in obj), None)\n        if isinstance(sd, torch.nn.Module):\n            return sd\n        if hp is not None and sd is not None:\n            m = cls(**dict(vars(hp) if hasattr(hp, "__dict__") and not isinstance(hp, dict) else hp))\n            m.load_state_dict(sd)\n            return m\n    raise RuntimeError(f"cannot load {path} as {cls.__name__}: Lightning said {lightning_err!r}; file holds "\n                       f"{sorted(obj)[:30] if isinstance(obj, dict) else type(obj)}")\n\n\ndef load(args):\n    return load_model(gen_model.FragGNN, args.gen), load_model(inten_model.IntenGNN, args.inten)\n\n\ndef to_peaks(s, k, prec, margin):\n    """(n, 2) array [mz, inten] -> (mz, it, above_n, above_share, above_off).\n\n    Drops peaks above prec + margin, then keeps the top-k by intensity, sorted by m/z, base = 1.\n    """\n    s = np.asarray(s, dtype=np.float64).reshape(-1, 2)\n    s = s[np.isfinite(s).all(axis=1) & (s[:, 1] > 0)]\n    above = s[:, 0] > prec + margin\n    above_n = int(above.sum())\n    above_share = float(s[above, 1].sum() / s[:, 1].sum()) if len(s) else 0.0\n    above_off = float(s[above, 0].max() - prec) if above_n else np.nan\n    s = s[~above]\n    if len(s) == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32), above_n, above_share, above_off\n    s = s[np.argsort(s[:, 1])[::-1][:k]]\n    s = s[np.argsort(s[:, 0])]\n    return (s[:, 0].astype(np.float32), (s[:, 1] / s[:, 1].max()).astype(np.float32),\n            above_n, above_share, above_off)\n\n\ndef predict_batch(model, rows, args):\n    return model.predict_mol(\n        [r["smiles"] for r in rows],\n        collision_eng=[float(r["ce"]) for r in rows],\n        precursor_mz=[float(r["precursor_mz"]) for r in rows],\n        adduct=[r["adduct"] for r in rows],\n        instrument=[r["instrument"] for r in rows],\n        threshold=args.threshold,\n        device=args.device,\n        max_nodes=args.max_nodes,\n        adduct_shift=False,\n        canonical_root_smi=True,\n        binned_out=False,\n    )["spec"]\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    gen, inten = load(args)\n\n    if args.hparams_only:\n        with open(args.out, "w") as f:\n            json.dump({"gen": _jsonable(gen.hparams), "inten": _jsonable(inten.hparams),\n                       "ion2onehot_pos": common.ion2onehot_pos,\n                       "instrument2onehot_pos": common.instrument2onehot_pos}, f, indent=1)\n        return\n\n    model = joint_model.JointModel(gen_model_obj=gen, inten_model_obj=inten).to(args.device)\n    model.eval()\n    model.freeze()\n    t_load = time.time() - t0\n\n    df = pd.read_csv(args.inp, sep="\t")\n    n = len(df)\n    peaks = [None] * n\n    err = [""] * n\n    above_n = np.zeros(n, np.int32)\n    above_share = np.zeros(n, np.float32)\n    above_off = np.full(n, np.nan, np.float32)\n\n    # canonicalise exactly as predict_smis.py does; reject what ms-pred cannot take\n    rows = []\n    for i, r in enumerate(df.to_dict("records")):\n        m = Chem.MolFromSmiles(r["smiles"]) if isinstance(r["smiles"], str) else None\n        ad = ADDUCT_MAP.get(r["adduct"], r["adduct"])\n        if m is None:\n            err[i] = "rdkit_parse"\n        elif ad not in common.ion2onehot_pos:\n            err[i] = f"adduct_unsupported:{r[\'adduct\']}"\n        elif r["instrument"] not in common.instrument2onehot_pos:\n            err[i] = f"instrument_unsupported:{r[\'instrument\']}"\n        elif not np.isfinite(r["ce"]):\n            err[i] = "ce_nan"\n        else:\n            rows.append(dict(r, i=i, smiles=Chem.MolToSmiles(m), adduct=ad))\n\n    t1 = time.time()\n    with torch.no_grad():\n        for b in range(0, len(rows), args.batch_size):\n            batch = rows[b:b + args.batch_size]\n            try:\n                specs = predict_batch(model, batch, args)\n            except Exception:\n                # isolate the failing molecule(s); keep the rest of the batch\n                specs = []\n                for r in batch:\n                    try:\n                        specs.append(predict_batch(model, [r], args)[0])\n                    except Exception as e:  # noqa: BLE001\n                        specs.append(None)\n                        err[r["i"]] = f"predict:{type(e).__name__}:{str(e)[:120]}"\n            for r, s in zip(batch, specs):\n                if s is not None:\n                    i = r["i"]\n                    mz_, it_, above_n[i], above_share[i], above_off[i] = to_peaks(\n                        s.detach().cpu().numpy(), args.sparse_k, float(r["precursor_mz"]), args.prec_margin)\n                    peaks[i] = (mz_, it_)\n    if args.device.startswith("cuda"):\n        torch.cuda.synchronize()\n    t_pred = time.time() - t1\n\n    empty = (np.empty(0, np.float32), np.empty(0, np.float32))\n    ok = np.array([p is not None and len(p[0]) > 0 for p in peaks])\n    for i in range(n):\n        if peaks[i] is not None and len(peaks[i][0]) == 0 and not err[i]:\n            err[i] = "empty_after_prec_cut" if above_n[i] else "empty_spectrum"\n    peaks = [p if p is not None else empty for p in peaks]\n    lens = np.array([len(p[0]) for p in peaks], dtype=np.int64)\n    off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in peaks]) if n else np.empty(0, np.float32)\n    it = np.concatenate([p[1] for p in peaks]) if n else np.empty(0, np.float32)\n    timing = dict(n=n, n_predicted=int(ok.sum()), t_load=t_load, t_pred=t_pred,\n                  mol_per_s=float(len(rows) / t_pred) if t_pred > 0 else None,\n                  device=args.device, batch_size=args.batch_size, prec_margin=args.prec_margin,\n                  gpu=torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu")\n    np.savez(args.out, off=off, mz=mz, it=it, ok=ok, err=np.array(err, dtype="U200"),\n             above_n=above_n, above_share=above_share, above_off=above_off, timing=json.dumps(timing))\n    print(json.dumps(timing))\n\n\nif __name__ == "__main__":\n    main()\n'
WORKER = WORK / "iceberg_worker.py"
WORKER.write_text(WORKER_SRC)

GEN, INTEN = os.path.join(UP, "ckpt", "gen.ckpt"), os.path.join(UP, "ckpt", "inten.ckpt")
assert sha1(GEN) == MAN["gen_sha1"] and sha1(INTEN) == MAN["inten_sha1"], "checkpoint SHA1 mismatch"
RESULT["sha1_ok"] = True
print(f"installed in {RESULT['install_s']} s; SHA1 OK")

## Request: 2,000 candidates from the pool ranges phase1 scores for np-examples molecules

In [ ]:
POOL = build_pool(COCO, TRAIN)
PM, PK, PS = POOL.mass.to_numpy(), POOL.k14.to_numpy(), POOL.smiles.to_numpy()
npx = pq.read_table(TRAIN, columns=["ingest_lib", "inchikey14", "adduct", "precursor_mz"],
                    filters=[("ingest_lib", "==", "enveda-np-examples")]).to_pandas()
ids = set()
for _, g in npx.groupby("inchikey14"):
    t = query_target(g.precursor_mz.to_numpy(), g.adduct.to_numpy())
    if np.isfinite(t):
        lo, hi = scored_range(PM, t)
        ids.update(range(lo, hi))
rng = np.random.default_rng(SEED)
pick = rng.choice(np.array(sorted(ids)), size=min(N_REQ, len(ids)), replace=False)
ads = np.array(["[M+H]+", "[M-H]-"], dtype=object)[np.arange(len(pick)) % 2]
ces = np.array([20.0, 40.0, 60.0])[(np.arange(len(pick)) // 2) % 3]
prec = np.array([precursor_of(m, a) for m, a in zip(PM[pick], ads)])
REQ = pd.DataFrame({"key": PK[pick], "smiles": PS[pick], "adduct": ads, "precursor_mz": prec,
                    "ce": to_model_ce(ces, prec, "ev"), "instrument": "QTOF"})
RESULT["request_mass"] = {k: round(float(v), 1) for k, v in pd.Series(PM[pick]).describe().items()}
print(f"{len(REQ):,} requests from {len(ids):,} scored candidates of {npx.inchikey14.nunique()} np-examples molecules; "
      f"mass {RESULT['request_mass']}")
del POOL

## Timed runs

In [ ]:
def launch(req, tag, gpu, bs):
    inp, out = WORK / f"req_{tag}.tsv", WORK / f"resp_{tag}.npz"
    write_request(req, inp)
    env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(worker_cmd(PY, WORKER, GEN, INTEN, inp, out, device="cuda", batch_size=bs),
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True), out, time.time()

def finish(p, out, t_start, n):
    so, se = p.communicate()
    wall = time.time() - t_start
    assert p.returncode == 0, se[-3000:]
    _, ok, err, timing = read_response(out, n)
    return dict(n=n, wall_s=wall, t_pred=timing["t_pred"], t_load=timing["t_load"], ok=float(np.mean(ok)),
                errors=err_counts(err), gpu=timing["gpu"], worker_mol_per_s=timing["mol_per_s"])

RUNS = {}
for bs in (64, 128):
    RUNS[f"1gpu_bs{bs}"] = r1 = finish(*launch(REQ, f"bs{bs}", 0, bs), len(REQ))
    r1.update(gpus=1, batch_size=bs, aggregate_mol_per_s=r1["n"] / r1["t_pred"],
              aggregate_wall_mol_per_s=r1["n"] / r1["wall_s"], startup_s=r1["wall_s"] - r1["t_pred"])
    print(f"1 GPU bs{bs}: {r1['aggregate_mol_per_s']:.1f} mol/s (pred), {r1['aggregate_wall_mol_per_s']:.1f} (wall); "
          f"startup {r1['startup_s']:.1f} s; ok {r1['ok']:.3f} {r1['errors']}")

best_bs = max((64, 128), key=lambda b: RUNS[f"1gpu_bs{b}"]["aggregate_mol_per_s"])
if len(GPUS) >= 2:
    half = len(REQ) // 2
    parts = [REQ.iloc[:half], REQ.iloc[half:]]
    procs = [launch(p, f"dual{g}", g, best_bs) for g, p in enumerate(parts)]
    rs = [finish(*pr, len(p)) for pr, p in zip(procs, parts)]
    n = sum(r["n"] for r in rs)
    RUNS[f"2gpu_bs{best_bs}"] = dict(
        gpus=2, batch_size=best_bs, n=n, parts=rs, ok=float(np.average([r["ok"] for r in rs], weights=[r["n"] for r in rs])),
        aggregate_mol_per_s=n / max(r["t_pred"] for r in rs), aggregate_wall_mol_per_s=n / max(r["wall_s"] for r in rs),
        startup_s=max(r["wall_s"] - r["t_pred"] for r in rs))
    d = RUNS[f"2gpu_bs{best_bs}"]
    print(f"2 GPUs bs{best_bs}: {d['aggregate_mol_per_s']:.1f} mol/s (pred), {d['aggregate_wall_mol_per_s']:.1f} (wall)")
else:
    print("only one GPU visible -- select 'GPU T4 x2' for the dual run; projecting with one GPU")

In [ ]:
best_key = max(RUNS, key=lambda k: RUNS[k]["aggregate_mol_per_s"])
B = RUNS[best_key]
RESULT.update(runs=RUNS, best=dict(config=best_key, gpus=B["gpus"], batch_size=B["batch_size"],
                                   aggregate_mol_per_s=B["aggregate_mol_per_s"],
                                   aggregate_wall_mol_per_s=B["aggregate_wall_mol_per_s"]),
              startup_s=max(r["startup_s"] for r in RUNS.values()),
              ok_rate=min(r["ok"] for r in RUNS.values()),
              finished=time.strftime("%Y-%m-%d %H:%M"))
Path("/kaggle/working/t4.json").write_text(json.dumps(RESULT, indent=1, default=str))
print("=" * 70)
print(json.dumps({k: RESULT[k] for k in ("gpus", "best", "startup_s", "ok_rate")}, indent=1))
print("=" * 70)
print("T4 step:", "PASS" if RESULT["ok_rate"] >= 0.99 else "CHECK -- ok rate below 0.99, read the error counts")
print("copy /kaggle/working/t4.json to Drive _simlib/grid/t4.json")